# R-style Analysis Demo

Two tracks in parallel:
- **Python track** (runs here): pandas + scipy + matplotlib doing exactly what R would do
- **R track** (paste into [Posit Cloud](https://posit.cloud) — free browser RStudio): identical analysis, R syntax

Data: `_overnight_results.jsonl` — 14+ hours of FNO CUDA inference on RTX 4060.

In [ ]:
import json, pathlib, re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scipy.stats as stats

# ── load JSONL ──────────────────────────────────────────────
# Some lines have concatenated objects — extract all with regex
raw = pathlib.Path('../_overnight_results.jsonl').read_text(errors='replace')
records = [json.loads(o) for o in re.findall(r'\{[^{}]+\}', raw)]
df = pd.DataFrame(records)
df['uptime_min'] = df['uptime_s'] / 60

print(f"Rows: {len(df)}")
print(df[['uptime_min','inf_per_sec','ms_per_batch','vram_mb','loss']].describe().round(3))

### R equivalent — paste into Posit Cloud
```r
library(jsonlite)
library(dplyr)

df <- stream_in(file("_overnight_results.jsonl")) %>%
  mutate(uptime_min = uptime_s / 60)

cat("Rows:", nrow(df), "\n")
df %>% select(uptime_min, inf_per_sec, ms_per_batch, vram_mb, loss) %>% summary()
```

In [ ]:
# ── summary stats table (publication-ready) ────────────────
summary = pd.DataFrame({
    'metric':   ['inf_per_sec', 'ms_per_batch', 'vram_mb', 'loss'],
    'mean':     [df['inf_per_sec'].mean(),  df['ms_per_batch'].mean(),
                 df['vram_mb'].mean(),       df['loss'].mean()],
    'sd':       [df['inf_per_sec'].std(),   df['ms_per_batch'].std(),
                 df['vram_mb'].std(),        df['loss'].std()],
    'median':   [df['inf_per_sec'].median(),df['ms_per_batch'].median(),
                 df['vram_mb'].median(),     df['loss'].median()],
    'min':      [df['inf_per_sec'].min(),   df['ms_per_batch'].min(),
                 df['vram_mb'].min(),        df['loss'].min()],
    'max':      [df['inf_per_sec'].max(),   df['ms_per_batch'].max(),
                 df['vram_mb'].max(),        df['loss'].max()],
}).set_index('metric').round(4)

print("\nPublication summary table:")
print(summary.to_string())

### R equivalent
```r
df %>%
  select(inf_per_sec, ms_per_batch, vram_mb, loss) %>%
  summarise(across(everything(),
    list(mean=mean, sd=sd, median=median, min=min, max=max),
    .names = "{.col}__{.fn}"
  )) %>%
  tidyr::pivot_longer(everything(),
    names_to  = c('metric','stat'), names_sep = '__') %>%
  tidyr::pivot_wider(names_from=stat, values_from=value)
```

In [ ]:
# ── 4-panel figure (IEEE style) ────────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('FNO Overnight Run — RTX 4060, 14h', fontsize=13, fontweight='bold')

# 1. Throughput over time
ax = axes[0,0]
ax.plot(df['uptime_min'], df['inf_per_sec'], lw=0.8, color='steelblue', alpha=0.7)
ax.axhline(df['inf_per_sec'].mean(), color='red', ls='--', lw=1.2, label=f"mean={df['inf_per_sec'].mean():.0f}")
ax.set_xlabel('Uptime (min)')
ax.set_ylabel('Inferences / sec')
ax.set_title('Throughput')
ax.legend(fontsize=8)

# 2. Latency over time
ax = axes[0,1]
ax.plot(df['uptime_min'], df['ms_per_batch'], lw=0.8, color='darkorange', alpha=0.7)
ax.axhline(df['ms_per_batch'].mean(), color='red', ls='--', lw=1.2, label=f"mean={df['ms_per_batch'].mean():.3f} ms")
ax.set_xlabel('Uptime (min)')
ax.set_ylabel('ms / batch')
ax.set_title('Batch Latency')
ax.legend(fontsize=8)

# 3. Histogram of throughput with normal fit
ax = axes[1,0]
ax.hist(df['inf_per_sec'], bins=60, density=True, color='steelblue', alpha=0.6, edgecolor='none')
mu, sigma = df['inf_per_sec'].mean(), df['inf_per_sec'].std()
x_fit = np.linspace(df['inf_per_sec'].min(), df['inf_per_sec'].max(), 200)
ax.plot(x_fit, stats.norm.pdf(x_fit, mu, sigma), 'r-', lw=2, label=f'N({mu:.0f}, {sigma:.0f})')
ax.set_xlabel('Inferences / sec')
ax.set_ylabel('Density')
ax.set_title('Throughput Distribution')
ax.legend(fontsize=8)

# 4. Loss over time (training phase visible)
ax = axes[1,1]
ax.semilogy(df['uptime_min'], df['loss'], lw=0.8, color='green', alpha=0.7)
ax.set_xlabel('Uptime (min)')
ax.set_ylabel('Loss (log scale)')
ax.set_title('Wrapped-phase Loss')
ax.set_ylim(bottom=1e-5)

plt.tight_layout()
plt.savefig('../_overnight_analysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: _overnight_analysis.png')

### R equivalent (ggplot2)
```r
library(ggplot2)
library(patchwork)  # combine panels like fig.subplots

p1 <- ggplot(df, aes(x=uptime_min, y=inf_per_sec)) +
  geom_line(color='steelblue', alpha=0.7, linewidth=0.4) +
  geom_hline(yintercept=mean(df$inf_per_sec), color='red', linetype='dashed') +
  labs(x='Uptime (min)', y='Inferences / sec', title='Throughput') +
  theme_classic()

p2 <- ggplot(df, aes(x=uptime_min, y=ms_per_batch)) +
  geom_line(color='darkorange', alpha=0.7, linewidth=0.4) +
  geom_hline(yintercept=mean(df$ms_per_batch), color='red', linetype='dashed') +
  labs(x='Uptime (min)', y='ms / batch', title='Batch Latency') +
  theme_classic()

p3 <- ggplot(df, aes(x=inf_per_sec)) +
  geom_histogram(aes(y=after_stat(density)), bins=60,
                 fill='steelblue', alpha=0.6) +
  stat_function(fun=dnorm,
    args=list(mean=mean(df$inf_per_sec), sd=sd(df$inf_per_sec)),
    color='red', linewidth=1) +
  labs(x='Inferences / sec', y='Density', title='Throughput Distribution') +
  theme_classic()

p4 <- ggplot(df, aes(x=uptime_min, y=loss)) +
  geom_line(color='darkgreen', alpha=0.7, linewidth=0.4) +
  scale_y_log10() +
  labs(x='Uptime (min)', y='Loss (log)', title='Wrapped-phase Loss') +
  theme_classic()

(p1 + p2) / (p3 + p4) +
  plot_annotation(title='FNO Overnight Run — RTX 4060, 14h')

ggsave('_overnight_analysis.pdf', width=12, height=8)  # vector PDF for IEEE
```

In [ ]:
# ── statistical tests (the R-specific value) ───────────────

# Q: did throughput drop significantly in the second half vs first half?
mid = len(df) // 2
first_half  = df['inf_per_sec'].iloc[:mid]
second_half = df['inf_per_sec'].iloc[mid:]

t_stat, p_val = stats.ttest_ind(first_half, second_half)
print("Two-sample t-test: first half vs second half throughput")
print(f"  first half  mean: {first_half.mean():.1f} +/- {first_half.std():.1f}")
print(f"  second half mean: {second_half.mean():.1f} +/- {second_half.std():.1f}")
print(f"  t = {t_stat:.3f},  p = {p_val:.2e}")
print(f"  -> {'SIGNIFICANT drop (thermal throttle?)' if p_val < 0.05 else 'no significant difference'}")
print()

# Shapiro-Wilk normality test on a sample
sample = df['inf_per_sec'].sample(min(200, len(df)), random_state=42)
sw_stat, sw_p = stats.shapiro(sample)
print("Shapiro-Wilk normality test (sample n=200):")
print(f"  W = {sw_stat:.4f},  p = {sw_p:.2e}")
print(f"  -> {'NOT normal (bimodal / skewed)' if sw_p < 0.05 else 'consistent with normal'}")
print()

# Pearson correlation: does higher throughput -> lower latency? (should be ~-1)
r, p_r = stats.pearsonr(df['inf_per_sec'], df['ms_per_batch'])
print(f"Pearson r(inf_per_sec, ms_per_batch) = {r:.4f}  p={p_r:.2e}")
print(f"  -> {'strong negative correlation (expected)' if r < -0.9 else 'weaker than expected'}")

### R equivalent
```r
# t-test
mid <- nrow(df) %/% 2
t.test(df$inf_per_sec[1:mid], df$inf_per_sec[(mid+1):nrow(df)])

# Shapiro-Wilk
shapiro.test(sample(df$inf_per_sec, 200))

# Pearson correlation
cor.test(df$inf_per_sec, df$ms_per_batch)

# R gives you: estimate, conf.int, t, df, p-value in one call
# This is what R is genuinely better at than Python for publication
```

---
**When to use R vs Python:**

| Task | Winner |
|------|--------|
| Statistical tests with proper output | R |
| Publication figures (ggplot2) | R |
| Survival analysis, mixed models | R |
| Signal processing, FFT, GS | Python |
| ML / PyTorch / CUDA | Python |
| Everything in this project | Python |